# VadCLIP

Although CLIP provides powerful visual-semantic representations learned from large-scale image-text pretraining, directly applying it to Weakly-Supervised Video Anomaly Detection (WSVAD) is challenging because anomaly detection depends on temporal context, requires effective use of semantic knowledge, and must operate under weak supervision where only video-level labels are available.

- **Challenge 1: Capturing Temporal Dependencies**
  - CLIP is primarily designed for image understanding and lacks an explicit mechanism to model temporal relationships across video frames.
  - In surveillance videos, anomalous events are often defined by how actions evolve over time rather than by a single frame.
  - The model must therefore learn both short-term dependencies between neighboring events and long-term temporal context across the entire video.

- **Challenge 2: Leveraging CLIP's Visual-Language Knowledge**
  - CLIP contains rich semantic knowledge acquired from large-scale image-text pairs, including concepts related to objects, actions, and events.
  - Simply using visual features ignores a significant portion of this pretrained knowledge.
  - The challenge is to effectively exploit the alignment between visual and textual representations so that semantic information can help distinguish normal and abnormal events, while preserving CLIP's general understanding capabilities.

- **Challenge 3: Adapting CLIP Under Weak Supervision**
  - WSVAD datasets typically provide labels only at the video level (e.g., normal or anomalous), without frame-level annotations indicating where anomalies occur.
  - Direct fine-tuning under such weak supervision can lead to noisy learning signals and may degrade the pretrained knowledge stored in CLIP.
  - The model must therefore identify the most relevant video segments for each label and learn anomaly representations without relying on precise temporal annotations.

To address the limitations of applying CLIP to WSVAD, VadCLIP introduces specialized components for temporal modeling, visual-language alignment, and weakly-supervised learning. The framework aims to preserve CLIP's pretrained semantic knowledge while adapting it to the anomaly detection setting.

- **Challenge 1: Temporal Modeling**
  - Introduces the **Local-Global Temporal Adapter (LGT-Adapter)**, a lightweight temporal modeling module.
    - **Local Temporal Adapter:** captures short-range temporal dependencies between neighboring events efficiently.
    - **Global Temporal Adapter:** captures broader temporal context and smooths feature representations with a small parameter overhead.

- **Challenge 2: Leveraging Visual-Language Knowledge**
  - VadCLIP besides using only visual features it encourage to also leverage textual features to preserve learned knowledge as much as possible
  - Uses a **dual-branch architecture**:
    - **C-branch:** performs binary classification using visual features.
    - **A-branch:** employs both visual and textual features for language-image alignment (build bridge between videos and video-level textual labels).
  - Enables both coarse-grained and fine-grained anomaly detection.
  - Introduces two prompt mechanisms:
    - **Learnable Prompt:** automatically adapts textual descriptions without handcrafted prompt engineering.
    - **Visual Prompt:** incorporates anomaly-related visual context into textual representations, improving discrimination between similar events (e.g., distinguishing a car accident from a fight).

<div>
    <img src='../images/VADCLIP.png' width="800">
</div>

- **Challenge 3: Weak Supervision**
  - Introduces **MIL-Align**, a Multiple Instance Learning alignment mechanism.
  - Selects the most matched video frames for each textual label.
  - Uses these highly matched frames to represent the entire video during language-visual alignment in A-branch.


> *Notes:* <br>
> ***Coarse-grained WSVAD:** predicts whether an entire video is normal or anomalous.* <br>
> ***Fine-grained WSVAD:** assigns anomaly scores to individual video segments, enabling temporal localization of anomalous events despite having only video-level supervision during training.*

## Method

### Local and Global Temporal Adapter

How to model temporal dependencies and bridge the gap between the image domain and video domain for CLIP.

**Local module**

For each frame an embedding $x$ of size $d = 512$ is generated for each frame by the CLIP model, for a total of $n$ frames then:

$$
X_{\textit{clip}} \in \mathbb{R}^{n \times d}
$$

In order to model temporal dependencias, a transformer encoder is used after the output of on top of frame-level features $X_{\textit{clip}}$. However, this encoder differs from the regular one as self-attention computation is only between local windows (frames that are close and overlaps), other wise would be computational expensive and some windows may not be relevant in the anomaly. In addtion, such an operation possesses local recep-
tive field like the convolution.

Frames:
```text
1 2 3 4 5 6 7 8 9 10
```

Temporal windows

```text
[1 2 3 4]
      [3 4 5 6]
            [5 6 7 8]
                  [7 8 9 10]
```

**Global module**

Models long-range temporal dependencies using a lightweight Graph Convolutional Network (GCN) built on top of locally enhanced frame features.

- Each video frame is treated as a graph node, and information is propagated through two types of relationships:
  - **Feature similarity (Hsim):** connects semantically similar frames regardless of temporal distance.
  - **Temporal distance (Hdis):** connects frames according to their relative positions in the video.
- By aggregating information through these graph connections, the module captures global video context with significantly lower computational cost than a full global self-attention mechanism.

```text
Local Module
↓
"What is happening near this frame?"
```

```text
Global Module
↓
"What other frames in the entire video look related to this frame?"
```

- **Feature Similarity Branch (Hsim):** constructs a graph based on frame-to-frame cosine similarity. Frames with similar visual-temporal representations are connected, allowing the GCN to propagate information between semantically related events, even when they are temporally distant. A thresholding operation removes weak connections, producing a sparse and more informative graph.

- **Position Distance Branch (Hdis):** constructs a graph based solely on temporal distance between frames. Frames that are closer in time receive stronger connections, while distant frames receive weaker connections. The parameter σ controls how quickly the influence decreases with temporal distance.
  - Unlike the Feature Similarity Branch, this branch does not use visual features; it preserves the temporal structure of the video by encoding positional relationships directly.
  - The final GCN combines both feature similarity (semantic relationships) and positional distance (temporal relationships), allowing the model to capture global context from complementary perspectives.
  - Residual connections are applied in both the local transformer and GCN layers to mitigate over-smoothing, preventing frame representations from becoming excessively similar after repeated information aggregation.

# Dual Branch

After temporal modeling, VadCLIP splits into two branches:

* **C-Branch (Classification Branch):**

  * Performs traditional binary anomaly classification.
  * Uses a FFN, a fully connected layer, and a Sigmoid activation to produce anomaly confidence scores (A).
  * Predicts whether each frame is normal or abnormal.

Example:

```text
Frame 1 → 0.05
Frame 2 → 0.08
Frame 3 → 0.92
Frame 4 → 0.87
```

* **A-Branch (Alignment Branch):**

  * Uses CLIP's frozen text encoder to transform anomaly labels (e.g., *fighting*, *riot*, *abuse*) into semantic class embeddings instead of one-hot vectors.
  * Computes similarities between frame-level visual features and class embeddings to create an alignment map.
  * Enables fine-grained anomaly recognition by associating frames with specific anomaly categories.

Example:

```text
Frame 25 → fighting
Frame 42 → riot
Frame 60 → abuse
```

# Class Embeddings

Instead of one-hot labels, VadCLIP encodes anomaly classes using CLIP's text encoder.

Example:

```text
"fighting"
   ↓
CLIP Text Encoder
   ↓
Class Embedding
```

* Class embeddings contain semantic information learned during CLIP pretraining.
* Similar anomaly concepts tend to be closer in the embedding space.

Example:

```text
fighting ↔ abuse  (closer)
fighting ↔ bicycle (farther)
```

# Learnable Prompt

Short anomaly labels may not sufficiently describe abnormal events.

Example:

```text
fighting
shooting
road accident
```

To enrich textual representations, VadCLIP introduces trainable context tokens inspired by CoOp.

Example:

```text
[c1] [c2] [c3]
    fighting
[c4] [c5] [c6]
```

* The context tokens are learned during training.
* The class token is placed in the middle of the sequence.
* The resulting prompt is fed into CLIP's text encoder to generate richer class embeddings.
* This allows the model to automatically learn task-specific textual context instead of relying on handcrafted prompts.

# Anomaly-Focus Visual Prompt

To further improve class embeddings, VadCLIP incorporates visual information from the current video.

* Uses anomaly confidence scores from the C-Branch as attention weights.
* Aggregates frame-level visual features into a video-level representation:

[
V = Norm(A^T X)
]

Example:

```text
Frame 1 → 0.05
Frame 2 → 0.10
Frame 3 → 0.90
Frame 4 → 0.85
```

The resulting visual prompt is dominated by Frames 3 and 4 because they are considered more anomalous.

Conceptually:

```text
Video Features
+
Anomaly Scores
↓
Visual Summary of Abnormal Segments
```

# Instance-Specific Class Embeddings

The anomaly-focus visual prompt is combined with the text embedding:

[
T = FFN(V + t_{out}) + t_{out}
]

* The FFN refines the combined representation.
* The skip connection preserves CLIP's original semantic knowledge.
* The final class embedding becomes video-dependent (instance-specific).

Example:

```text
"fighting"
+
visual context from Video A
↓
Class Embedding A

"fighting"
+
visual context from Video B
↓
Class Embedding B
```

Although both videos contain the same anomaly category, their final class embeddings may differ because their visual contexts are different.

# Alignment Map

After obtaining:

* Frame-level visual features (X)
* Instance-specific class embeddings (T)

VadCLIP computes similarities between them to build the alignment map (M).

Example:

| Frame | Fighting | Riot | Abuse |
| ----- | -------- | ---- | ----- |
| F1    | 0.12     | 0.08 | 0.05  |
| F2    | 0.91     | 0.14 | 0.09  |
| F3    | 0.87     | 0.18 | 0.11  |

* Each value indicates how well a frame matches a specific anomaly class.
* The alignment map bridges visual representations and semantic anomaly labels.
* This mechanism enables fine-grained anomaly recognition and localization.
